## 1. Setup — install dependencies and mount Drive

In [ ]:
!pip install torch_geometric -q

from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.9 MB/s eta 0:00:00
Mounted at /content/drive


## 2. Load the preprocessed PrimeKG-CL graph
Loads the cleaned, indexed heterogeneous graph built in the preprocessing notebook.

In [ ]:
import torch
import pandas as pd
import numpy as np
from torch_geometric.data import HeteroData

save_dir = "/content/drive/MyDrive/PharmaGraph_Preprocessing"

data = torch.load(f"{save_dir}/pharmagraph_hetero_t0.pt", weights_only=False)
nodes_typed = pd.read_csv(f"{save_dir}/entities_local_idx.csv")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = data.to(device)

print(data)

HeteroData(
  anatomy={
    num_nodes=14033,
    x=[14033, 1],
  },
  biological_process={
    num_nodes=28642,
    x=[28642, 1],
  },
  cellular_component={
    num_nodes=4176,
    x=[4176, 1],
  },
  disease={
    num_nodes=17080,
    x=[17080, 1],
  },
  drug={
    num_nodes=7957,
    x=[7957, 1],
  },
  effect/phenotype={
    num_nodes=15311,
    x=[15311, 1],
  },
  exposure={
    num_nodes=818,
    x=[818, 1],
  },
  gene/protein={
    num_nodes=27610,
    x=[27610, 1],
  },
  molecular_function={
    num_nodes=11169,
    x=[11169, 1],
  },
  pathway={
    num_nodes=2516,
    x=[2516, 1],
  },
  (anatomy, anatomy_anatomy, anatomy)={ edge_index=[2, 28064] },
  (anatomy, anatomy_protein_absent, gene/protein)={ edge_index=[2, 19887] },
  (gene/protein, anatomy_protein_absent, anatomy)={ edge_index=[2, 19887] },
  (anatomy, anatomy_protein_present, gene/protein)={ edge_index=[2, 1518203] },
  (gene/protein, anatomy_protein_present, anatomy)={ edge_index=[2, 1518203] },
  (biological_

## 3. Add real molecular features for drugs
Replaces placeholder (all-ones) drug features with Morgan fingerprints fetched from PubChem.

In [ ]:
drug_fingerprints = np.load(f"{save_dir}/drug_fingerprints.npy", allow_pickle=True).item()

drug_nodes = nodes_typed[nodes_typed["node_type"] == "drug"].sort_values("local_idx")
FP_SIZE = 256
drug_feature_matrix = np.zeros((len(drug_nodes), FP_SIZE), dtype=np.float32)

found = 0
for _, row in drug_nodes.iterrows():
    if row["node_id"] in drug_fingerprints:
        drug_feature_matrix[row["local_idx"]] = drug_fingerprints[row["node_id"]]
        found += 1

print(f"Drugs with real molecular features: {found} / {len(drug_nodes)}")

data["drug"].x = torch.tensor(drug_feature_matrix, dtype=torch.float, device=device)

Drugs with real molecular features: 7058 / 7957


## 4. Define the task — positive and negative labels
Positives come from `indication` edges. Hard negatives come from `contraindication` edges
(after removing the small number that overlap with positives), balanced 2:1 with random negatives.

In [ ]:
indication_edge_index = data["drug", "indication", "disease"].edge_index
drug_idx = indication_edge_index[0]
disease_idx = indication_edge_index[1]

contra_edge_index = data["drug", "contraindication", "disease"].edge_index

positive_pairs = set(zip(drug_idx.tolist(), disease_idx.tolist()))
hard_neg_pairs = set(zip(contra_edge_index[0].tolist(), contra_edge_index[1].tolist()))
clean_hard_neg_pairs = hard_neg_pairs - positive_pairs

n_drugs = data["drug"].num_nodes
n_diseases = data["disease"].num_nodes

def sample_random_negatives(n_samples, n_drugs, n_diseases, positive_pairs, seed=42):
    rng = np.random.default_rng(seed)
    negatives = []
    while len(negatives) < n_samples:
        d = rng.integers(0, n_drugs)
        dis = rng.integers(0, n_diseases)
        if (d, dis) not in positive_pairs:
            negatives.append((d, dis))
    return negatives

random_negatives = sample_random_negatives(len(positive_pairs), n_drugs, n_diseases, positive_pairs)

rng = np.random.default_rng(42)
NEG_RATIO = 2
n_positives = len(positive_pairs)
n_neg_target = n_positives * NEG_RATIO
n_hard_target = min(len(clean_hard_neg_pairs), n_neg_target // 2)
n_random_target = n_neg_target - n_hard_target

clean_hard_neg_list = list(clean_hard_neg_pairs)
sampled_hard_idx = rng.choice(len(clean_hard_neg_list), size=n_hard_target, replace=False)
sampled_hard = [clean_hard_neg_list[i] for i in sampled_hard_idx]
sampled_random = random_negatives[:n_random_target]

final_neg_drug = [p[0] for p in sampled_hard] + [p[0] for p in sampled_random]
final_neg_disease = [p[1] for p in sampled_hard] + [p[1] for p in sampled_random]

print(f"Positives: {n_positives} | Negatives: {len(final_neg_drug)} (ratio {len(final_neg_drug)/n_positives:.1f}:1)")

Positives: 9388 | Negatives: 18776 (ratio 2.0:1)


## 5. Leak-free train / validation / test split
Splits drug-disease pairs 70/15/15. Both edge directions (drug→disease and disease→drug) for
any held-out pair are removed from the training graph to prevent leakage.

In [ ]:
from sklearn.model_selection import train_test_split
import copy

pos_drug_arr = np.array(list(zip(*positive_pairs))[0])
pos_disease_arr = np.array(list(zip(*positive_pairs))[1])

pos_train_idx, pos_temp_idx = train_test_split(np.arange(len(pos_drug_arr)), test_size=0.30, random_state=42)
pos_val_idx, pos_test_idx = train_test_split(pos_temp_idx, test_size=0.50, random_state=42)

neg_drug_arr = np.array(final_neg_drug)
neg_disease_arr = np.array(final_neg_disease)

neg_train_idx, neg_temp_idx = train_test_split(np.arange(len(neg_drug_arr)), test_size=0.30, random_state=42)
neg_val_idx, neg_test_idx = train_test_split(neg_temp_idx, test_size=0.50, random_state=42)

val_test_pairs = set()
for idx in np.concatenate([pos_val_idx, pos_test_idx]):
    val_test_pairs.add((pos_drug_arr[idx], pos_disease_arr[idx]))

def build_keep_mask(edge_index, hide_pairs, drug_first=True):
    mask = torch.ones(edge_index.shape[1], dtype=torch.bool)
    for i in range(edge_index.shape[1]):
        d = edge_index[0, i].item() if drug_first else edge_index[1, i].item()
        dis = edge_index[1, i].item() if drug_first else edge_index[0, i].item()
        if (d, dis) in hide_pairs:
            mask[i] = False
    return mask

train_graph = copy.copy(data)

ei_fwd = data["drug", "indication", "disease"].edge_index
mask_fwd = build_keep_mask(ei_fwd, val_test_pairs, drug_first=True)
train_graph["drug", "indication", "disease"].edge_index = ei_fwd[:, mask_fwd]

ei_rev = data["disease", "indication", "drug"].edge_index
mask_rev = build_keep_mask(ei_rev, val_test_pairs, drug_first=False)
train_graph["disease", "indication", "drug"].edge_index = ei_rev[:, mask_rev]

print("Indication edges after masking (train only):", train_graph["drug", "indication", "disease"].edge_index.shape[1])

Indication edges after masking (train only): 6571


## 6. Model — 2-layer heterogeneous GraphSAGE
Defines the encoder (per-relation SAGEConv layers) and the dot-product scoring function
used to rank drug-disease candidates.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import HeteroConv, SAGEConv

class HeteroGraphSAGE(nn.Module):
    def __init__(self, node_types, edge_types, hidden_channels=128, out_channels=128):
        super().__init__()
        self.conv1 = HeteroConv({
            edge_type: SAGEConv((-1, -1), hidden_channels)
            for edge_type in edge_types
        }, aggr='sum')
        self.conv2 = HeteroConv({
            edge_type: SAGEConv((-1, -1), out_channels)
            for edge_type in edge_types
        }, aggr='sum')

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {key: F.relu(x) for key, x in x_dict.items()}
        x_dict = self.conv2(x_dict, edge_index_dict)
        return x_dict

def score_pairs(drug_emb, disease_emb, drug_idx, disease_idx):
    d = drug_emb[drug_idx]
    dis = disease_emb[disease_idx]
    return (d * dis).sum(dim=-1)

model = HeteroGraphSAGE(train_graph.node_types, train_graph.edge_types, hidden_channels=128, out_channels=128).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
margin_loss_fn = nn.MarginRankingLoss(margin=1.0)

with torch.no_grad():
    out = model(train_graph.x_dict, train_graph.edge_index_dict)
print("Drug embedding shape:", out["drug"].shape)

Drug embedding shape: torch.Size([7957, 128])


## 7. Prepare train / validation / test tensors

In [ ]:
pos_train_drug = torch.tensor(pos_drug_arr[pos_train_idx], dtype=torch.long, device=device)
pos_train_disease = torch.tensor(pos_disease_arr[pos_train_idx], dtype=torch.long, device=device)
neg_train_drug = torch.tensor(neg_drug_arr[neg_train_idx], dtype=torch.long, device=device)
neg_train_disease = torch.tensor(neg_disease_arr[neg_train_idx], dtype=torch.long, device=device)

val_drug = torch.tensor(np.concatenate([pos_drug_arr[pos_val_idx], neg_drug_arr[neg_val_idx]]), dtype=torch.long, device=device)
val_disease = torch.tensor(np.concatenate([pos_disease_arr[pos_val_idx], neg_disease_arr[neg_val_idx]]), dtype=torch.long, device=device)
val_labels = torch.tensor(np.concatenate([np.ones(len(pos_val_idx)), np.zeros(len(neg_val_idx))]), dtype=torch.float, device=device)

test_drug = torch.tensor(np.concatenate([pos_drug_arr[pos_test_idx], neg_drug_arr[neg_test_idx]]), dtype=torch.long, device=device)
test_disease = torch.tensor(np.concatenate([pos_disease_arr[pos_test_idx], neg_disease_arr[neg_test_idx]]), dtype=torch.long, device=device)
test_labels = torch.tensor(np.concatenate([np.ones(len(pos_test_idx)), np.zeros(len(neg_test_idx))]), dtype=torch.float, device=device)

## 8. Training and evaluation functions
Margin ranking loss for training; AUROC/AUPRC for evaluation.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

def train_epoch():
    model.train()
    optimizer.zero_grad()
    out = model(train_graph.x_dict, train_graph.edge_index_dict)
    drug_emb = out["drug"]
    disease_emb = out["disease"]

    pos_scores = score_pairs(drug_emb, disease_emb, pos_train_drug, pos_train_disease)
    n_neg = len(neg_train_drug)
    n_pos = len(pos_train_drug)
    repeat_factor = (n_neg // n_pos) + 1
    pos_scores_matched = pos_scores.repeat(repeat_factor)[:n_neg]

    neg_scores = score_pairs(drug_emb, disease_emb, neg_train_drug, neg_train_disease)
    target = torch.ones(n_neg, device=device)
    loss = margin_loss_fn(pos_scores_matched, neg_scores, target)

    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate(split_drug, split_disease, split_labels):
    model.eval()
    with torch.no_grad():
        out = model(train_graph.x_dict, train_graph.edge_index_dict)
        drug_emb = out["drug"]
        disease_emb = out["disease"]
        scores = score_pairs(drug_emb, disease_emb, split_drug, split_disease)
        probs = torch.sigmoid(scores).cpu().numpy()
    labels = split_labels.cpu().numpy()
    return roc_auc_score(labels, probs), average_precision_score(labels, probs)

## 9. Train with early stopping
Includes dropout and weight decay to control overfitting — an earlier run without
regularization performed worse than the placeholder-feature baseline.

In [ ]:
best_auprc = 0
best_state = None
patience = 10
epochs_no_improve = 0

for epoch in range(1, 101):
    loss = train_epoch()
    if epoch % 5 == 0 or epoch == 1:
        auroc, auprc = evaluate(val_drug, val_disease, val_labels)
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | Val AUROC: {auroc:.4f} | Val AUPRC: {auprc:.4f}")
        if auprc > best_auprc:
            best_auprc = auprc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_state)
print(f"\nBest Val AUPRC: {best_auprc:.4f}")

Epoch   1 | Loss: 24.6805 | Val AUROC: 0.6430 | Val AUPRC: 0.4719
Epoch   5 | Loss: 9.8545 | Val AUROC: 0.7218 | Val AUPRC: 0.4771
Epoch  10 | Loss: 4.5705 | Val AUROC: 0.7252 | Val AUPRC: 0.4801
Epoch  15 | Loss: 2.2304 | Val AUROC: 0.7265 | Val AUPRC: 0.4786
Epoch  20 | Loss: 1.3251 | Val AUROC: 0.7209 | Val AUPRC: 0.4770
Epoch  25 | Loss: 0.9349 | Val AUROC: 0.7325 | Val AUPRC: 0.4895
Epoch  30 | Loss: 0.4960 | Val AUROC: 0.7292 | Val AUPRC: 0.4836
Epoch  35 | Loss: 0.3199 | Val AUROC: 0.7286 | Val AUPRC: 0.4818
Epoch  40 | Loss: 0.1909 | Val AUROC: 0.7264 | Val AUPRC: 0.4802
Epoch  45 | Loss: 0.1015 | Val AUROC: 0.7271 | Val AUPRC: 0.4807
Epoch  50 | Loss: 0.0475 | Val AUROC: 0.7305 | Val AUPRC: 0.4831
Epoch  55 | Loss: 0.0187 | Val AUROC: 0.7305 | Val AUPRC: 0.4832
Epoch  60 | Loss: 0.0054 | Val AUROC: 0.7310 | Val AUPRC: 0.4836
Epoch  65 | Loss: 0.0011 | Val AUROC: 0.7302 | Val AUPRC: 0.4828
Epoch  70 | Loss: 0.0005 | Val AUROC: 0.7293 | Val AUPRC: 0.4822
Epoch  75 | Loss: 0.0000

## 10. Final test evaluation and checkpoint
Final result: Test AUROC = 0.9257, Test AUPRC = 0.8818.

In [ ]:
test_auroc, test_auprc = evaluate(test_drug, test_disease, test_labels)

print("=" * 50)
print("FINAL RESULTS — GraphSAGE + real molecular features")
print("=" * 50)
print(f"Test AUROC: {test_auroc:.4f}")
print(f"Test AUPRC: {test_auprc:.4f}")

torch.save({
    "model_state": best_state,
    "test_auroc": test_auroc,
    "test_auprc": test_auprc,
    "val_auprc": best_auprc,
    "notes": "v2: 2-layer heterogeneous GraphSAGE, real Morgan fingerprints for drugs (256-dim), indication vs contraindication+random negatives (2:1)"
}, f"{save_dir}/models/graphsage_v2_real_features.pt")

print("\nSaved to Drive.")

FINAL RESULTS — GraphSAGE + real molecular features
Test AUROC: 0.7288
Test AUPRC: 0.4857

Saved to Drive.


In [ ]:
model = HeteroGraphSAGE(train_graph.node_types, train_graph.edge_types, hidden_channels=128, out_channels=128).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # weight decay جديد
margin_loss_fn = nn.MarginRankingLoss(margin=1.0)

dropout = nn.Dropout(p=0.3)

def train_epoch():
    model.train()
    optimizer.zero_grad()
    out = model(train_graph.x_dict, train_graph.edge_index_dict)
    drug_emb = dropout(out["drug"])
    disease_emb = dropout(out["disease"])

    pos_scores = score_pairs(drug_emb, disease_emb, pos_train_drug, pos_train_disease)
    n_neg = len(neg_train_drug)
    n_pos = len(pos_train_drug)
    repeat_factor = (n_neg // n_pos) + 1
    pos_scores_matched = pos_scores.repeat(repeat_factor)[:n_neg]

    neg_scores = score_pairs(drug_emb, disease_emb, neg_train_drug, neg_train_disease)
    target = torch.ones(n_neg, device=device)
    loss = margin_loss_fn(pos_scores_matched, neg_scores, target)

    loss.backward()
    optimizer.step()
    return loss.item()

In [ ]:
best_auprc = 0
best_state = None
patience = 10
epochs_no_improve = 0

for epoch in range(1, 101):
    loss = train_epoch()
    if epoch % 5 == 0 or epoch == 1:
        auroc, auprc = evaluate(val_drug, val_disease, val_labels)
        print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | Val AUROC: {auroc:.4f} | Val AUPRC: {auprc:.4f}")
        if auprc > best_auprc:
            best_auprc = auprc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_state)
print(f"\nBest Val AUPRC: {best_auprc:.4f}")

Epoch   1 | Loss: 42.6465 | Val AUROC: 0.6368 | Val AUPRC: 0.4080
Epoch   5 | Loss: 5.2584 | Val AUROC: 0.7973 | Val AUPRC: 0.5619
Epoch  10 | Loss: 2.9903 | Val AUROC: 0.8922 | Val AUPRC: 0.7997
Epoch  15 | Loss: 1.7993 | Val AUROC: 0.8798 | Val AUPRC: 0.7249
Epoch  20 | Loss: 1.3552 | Val AUROC: 0.8978 | Val AUPRC: 0.8582
Epoch  25 | Loss: 0.8370 | Val AUROC: 0.8982 | Val AUPRC: 0.7942
Epoch  30 | Loss: 0.4872 | Val AUROC: 0.9143 | Val AUPRC: 0.8538
Epoch  35 | Loss: 0.3813 | Val AUROC: 0.9160 | Val AUPRC: 0.8696
Epoch  40 | Loss: 0.2781 | Val AUROC: 0.9171 | Val AUPRC: 0.8587
Epoch  45 | Loss: 0.2032 | Val AUROC: 0.9216 | Val AUPRC: 0.8759
Epoch  50 | Loss: 0.1622 | Val AUROC: 0.9252 | Val AUPRC: 0.8786
Epoch  55 | Loss: 0.1185 | Val AUROC: 0.9265 | Val AUPRC: 0.8726
Epoch  60 | Loss: 0.0727 | Val AUROC: 0.9256 | Val AUPRC: 0.8814
Epoch  65 | Loss: 0.0728 | Val AUROC: 0.9281 | Val AUPRC: 0.8907
Epoch  70 | Loss: 0.0622 | Val AUROC: 0.9310 | Val AUPRC: 0.8922
Epoch  75 | Loss: 0.0559

In [ ]:
test_auroc_v3, test_auprc_v3 = evaluate(test_drug, test_disease, test_labels)

print("=" * 60)
print("FINAL COMPARISON — all three versions")
print("=" * 60)
print(f"{'Version':<35}{'Test AUROC':<15}{'Test AUPRC':<15}")
print(f"{'v1: placeholder features':<35}{0.7918:<15}{0.5542:<15}")
print(f"{'v2: real features, no regularization':<35}{0.7288:<15}{0.4857:<15}")
print(f"{'v3: real features + dropout/decay':<35}{test_auroc_v3:<15.4f}{test_auprc_v3:<15.4f}")

FINAL COMPARISON — all three versions
Version                            Test AUROC     Test AUPRC     
v1: placeholder features           0.7918         0.5542         
v2: real features, no regularization0.7288         0.4857         
v3: real features + dropout/decay  0.9257         0.8818         


## Sanity Check: Is the 0.93 AUROC real, or is the model just copying chemically-similar drugs?

We noticed a large jump in performance after adding real molecular fingerprints (AUROC went
from 0.79 -> 0.93). Before trusting this number, we ran two checks:

1. **Leakage check**: confirmed no drug-disease pair appears in both train and test, in either
   direction. No leakage at the pair level.

2. **Chemical similarity check**: found that 83% of test drugs have a near-identical fingerprint
   ("twin") already in the training set. This raised a real concern — maybe the model is just
   memorizing "this drug looks like that drug, so it probably works the same way" instead of
   learning real biological structure.

**Stratified evaluation result:**
- AUROC on drugs WITH a chemical twin in train: 0.9258
- AUROC on "novel" drugs WITHOUT a chemical twin: 0.9008

The gap is small (~0.025). If the model were only copying chemical twins, novel-drug AUROC
would have collapsed toward ~0.5-0.6. It didn't — so the model is genuinely learning from graph
structure (targets, pathways, proteins), not just chemical lookup.

**Conclusion: the 0.93 AUROC baseline is trustworthy.** Below is the reusable check code, in
case we want to re-run this same audit after adding the Open Targets evidence layer.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score

def audit_chemical_twin_bias(model, train_graph, drug_feature_matrix,
                              train_drug_ids, test_drug_ids,
                              test_pair_drug_ids, test_pair_disease_ids, test_pair_labels,
                              similarity_threshold=0.95):
    """
    Checks whether model performance depends on test drugs having a
    near-identical fingerprint ("twin") in the training set. Returns
    stratified AUROC for twin vs. novel drugs.
    """
    # find which test drugs have a close chemical twin in train
    train_fps = drug_feature_matrix[train_drug_ids]
    unique_test_ids = np.unique(test_pair_drug_ids)
    sim_matrix = cosine_similarity(drug_feature_matrix[unique_test_ids], train_fps)
    max_sim = sim_matrix.max(axis=1)
    twin_lookup = dict(zip(unique_test_ids, max_sim > similarity_threshold))

    has_twin = np.array([twin_lookup[d] for d in test_pair_drug_ids])

    # score every test pair
    model.eval()
    with torch.no_grad():
        out = model(train_graph.x_dict, train_graph.edge_index_dict)
        scores = score_pairs(
            out["drug"], out["disease"],
            torch.tensor(test_pair_drug_ids, device=device),
            torch.tensor(test_pair_disease_ids, device=device),
        )
        probs = torch.sigmoid(scores).cpu().numpy()

    auroc_twin = roc_auc_score(test_pair_labels[has_twin], probs[has_twin])
    auroc_novel = roc_auc_score(test_pair_labels[~has_twin], probs[~has_twin])

    print(f"Test pairs with a chemical twin:   {has_twin.sum():5d} ({has_twin.mean()*100:.1f}%) | AUROC = {auroc_twin:.4f}")
    print(f"Test pairs without a twin (novel): {(~has_twin).sum():5d} ({(1-has_twin.mean())*100:.1f}%) | AUROC = {auroc_novel:.4f}")
    print(f"Gap (twin - novel): {auroc_twin - auroc_novel:+.4f}  (small gap = model isn't just copying twins)")

    return auroc_twin, auroc_novel


# run it on our current baseline
test_pair_drug_ids = np.concatenate([pos_drug_arr[pos_test_idx], neg_drug_arr[neg_test_idx]])
test_pair_disease_ids = np.concatenate([pos_disease_arr[pos_test_idx], neg_disease_arr[neg_test_idx]])
test_pair_labels = np.concatenate([np.ones(len(pos_test_idx)), np.zeros(len(neg_test_idx))])

audit_chemical_twin_bias(
    model, train_graph, drug_feature_matrix,
    train_drug_ids, test_drug_ids,
    test_pair_drug_ids, test_pair_disease_ids, test_pair_labels,
)

Test pairs with a chemical twin:    3521 (83.3%) | AUROC = 0.9258
Test pairs without a twin (novel):   705 (16.7%) | AUROC = 0.9008
Gap (twin - novel): +0.0249  (small gap = model isn't just copying twins)


(np.float64(0.9257577202270155), np.float64(0.9008343715166218))